Persiapan

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, row_number
from pyspark.sql.window import Window

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas7_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap! Versi spark: ", spark.version)

SparkSession siap! Versi spark:  3.5.9


Menyiapkan Dataset

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


A. EXTRACT

In [5]:
# Membaca ketiga sumber data dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Data berhasil dimuat dari HDFS.")
print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Data berhasil dimuat dari HDFS.
Transaksi: 4000 | Cabang: 3 | Menu: 15


B. TRANSFORM — Join & Kolom Turunan

In [7]:
# Menggabungkan ketiga tabel
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner")

# Menambahkan kolom total_penjualan (qty x harga)
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


C. TRANSFORM — Window Function

In [8]:
# Menghitung total penjualan per cabang dan per menu
ringkasan_menu = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Membuat window yang dipartisi berdasarkan nama_cabang dan diurutkan berdasarkan total_penjualan menurun
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# Mengambil top-2 menu terlaris di setiap cabang
top2_menu = ringkasan_menu.withColumn("peringkat", row_number().over(window_cabang)) \
                          .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



D. TRANSFORM — Spark SQL

In [9]:
# Membuat temporary view untuk Spark SQL
df_gabungan.createOrReplaceTempView("kopi_nusantara")

# Kueri untuk melihat kontribusi total penjualan per kepala_barista
kontribusi_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



E. LOAD

In [10]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Menyimpan ke format Parquet, dipartisi berdasarkan kategori_menu
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(output_path)

# Verifikasi dengan membaca kembali data yang baru saja disimpan
df_laporan = spark.read.parquet(output_path)
print("Verifikasi Load - Total baris dari Parquet:", df_laporan.count())

Verifikasi Load - Total baris dari Parquet: 4000


F. Ringkasan Eksekutif & Rekomendasi

In [12]:
# Membuat ringkasan eksekutif
ringkasan_eksekutif = df_laporan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif.show(truncate=False)

+-------------------------+----------+--------------+------------------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000                |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000                |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000                |1305            |64862.06896551724        |
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+



Dari data penjualan yang kita olah, kita bisa melihat dengan jelas bagaimana performa masing-masing cabang Kopi Nusantara. Juara kita saat ini adalah Kopi Nusantara Semarang. Cabang ini berhasil meraih total pendapatan paling tinggi, pelanggannya ramai, dan rata-rata uang yang dihabiskan tiap pelanggan juga bagus. Artinya, layanan di cabang ini sudah sangat pas dan disukai banyak orang.

Di sisi lain, kita perlu memberi perhatian khusus pada Kopi Nusantara Yogyakarta, karena angka penjualannya masih berada di urutan paling bawah. Tapi tenang, kita punya strategi untuk meningkatkan penjualannya. Dari data sebelumnya, kita tahu kalau pelanggan di cabang ini paling suka pesan Menu-12 dan Menu-15.

Nah, saran saya, mari kita jadikan dua menu andalan ini sebagai daya tarik utama! Kita bisa membuat paket bundling atau promo khusus. Misalnya, pelanggan yang beli Menu-12 bisa dapat diskon untuk menu lain yang penjualannya agak sepi. Dengan mempromosikan menu yang sudah pasti mereka suka, kita bisa menarik lebih banyak pelanggan datang sekaligus membuat mereka berbelanja lebih banyak.